In [8]:
#!python -m pip install -U python-woc pandas matplotlib
#import sys
# 1. Uninstall the incorrect package
#!{sys.executable} -m pip uninstall -y woc

# 2. Install the correct World of Code package
#!{sys.executable} -m pip install python-woc
# please clear the output of this cell in your notebook before checking it in

In [9]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['pmelchior_scarlet']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,002c2efe928bc279bb968188dcd07c1953d8d0f6,pmelchior_scarlet


In [10]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

100%|██████████████████████████████████████████████████████████████| 163/163 [02:46<00:00,  1.02s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,002c2efe928bc279bb968188dcd07c1953d8d0f6,12a4a2e0d1d74a6bec2a5be51309d88c71050f94,[cdc29625cf087396c8732e2576dd87d80be94979],herjy <remy.joseph11@gmail.com>,1586978476,-0400,herjy <remy.joseph11@gmail.com>,1586978476,-0400,Cleaned unnecessary dependencies\n,002c2efe928bc279bb968188dcd07c1953d8d0f6,pmelchior_scarlet
1,002d22cec20062a7451e0ad9eb1bd6d16340b152,50e69bfca1136b85f1055e0735f61653c6a41723,[f678733ee661c8f013914375f5b10eddd9f9c6ad],herjy <remy.joseph11@gmail.com>,1588732939,-0400,herjy <remy.joseph11@gmail.com>,1588732939,-0400,multiresolution notebook reduced runtime\n,002d22cec20062a7451e0ad9eb1bd6d16340b152,pmelchior_scarlet


In [11]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '002c2efe928bc279bb968188dcd07c1953d8d0f6', 'tree': '12a4a2e0d1d74a6bec2a5be51309d88c71050f94', 'parent': ['cdc29625cf087396c8732e2576dd87d80be94979'], 'author': 'herjy <remy.joseph11@gmail.com>', 'author_time': 1586978476, 'author_tz': '-0400', 'committer': 'herjy <remy.joseph11@gmail.com>', 'committer_time': 1586978476, 'committer_tz': '-0400', 'message': 'Cleaned unnecessary dependencies\n'}


In [12]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [13]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,pmelchior_scarlet,002c2efe928bc279bb968188dcd07c1953d8d0f6,herjy <remy.joseph11@gmail.com>,1586978476,Cleaned unnecessary dependencies\n


In [14]:
#mode a means append, so you have all your projects in the same file
yournetid='asangiul'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github